# Instruction Fine-Tuning Notebook
## Stage 2: Q&A Training

This notebook performs instruction fine-tuning on question-answer pairs to teach the model how to respond to user queries.

## Step 1: Install Required Libraries

In [ ]:
!pip install -q torch transformers datasets peft bitsandbytes accelerate unsloth[colab-new] -U

## Step 2: Import Libraries

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import Dataset
from peft import LoraConfig, get_peft_model
import json
import os

print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

## Step 3: Load Instruction Dataset

In [ ]:
# Load instruction dataset
dataset_path = 'course-doubt-assistant/data/instruction_dataset.jsonl'

data = []
with open(dataset_path, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

print(f"Loaded {len(data)} instruction-response pairs")
print(f"\nFirst example:")
print(f"  Instruction: {data[0]['instruction']}")
print(f"  Response: {data[0]['response'][:100]}...")

## Step 4: Format Instruction Dataset

In [ ]:
# Create formatted text for training
def format_instruction(example):
    return {
        'text': f"### Instruction:
{example['instruction']}
### Response:
{example['response']}"
    }

formatted_data = [format_instruction(d) for d in data]

print("First formatted example:")
print(formatted_data[0]['text'])

## Step 5: Create Dataset Object

In [ ]:
# Create dataset
dataset = Dataset.from_dict({'text': [d['text'] for d in formatted_data]})

print(f"Dataset size: {len(dataset)}")
print(f"Dataset columns: {dataset.column_names}")

## Step 6: Load Model and Tokenizer

In [ ]:
# Model configuration
MODEL_NAME = 'unsloth/tinyllama-bnb-4bit'

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16
)

print(f"Model loaded: {MODEL_NAME}")

## Step 7: Configure LoRA for Instruction Tuning

In [ ]:
# LoRA Configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'v_proj'],
    modules_to_save=['lm_head']
)

# Apply LoRA
model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,}")
print(f"Total params: {total_params:,}")
print(f"Trainable %: {100 * trainable_params / total_params:.2f}%")

## Step 8: Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

# Tokenize
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['text']
)

print(f"Tokenized dataset size: {len(tokenized_dataset)}")

## Step 9: Configure Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir='./outputs/instruction_ft',
    num_train_epochs=3,                    # 3 epochs for Q&A
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,                    # Lower LR for fine-tuning
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=5,
    save_steps=25,
    save_total_limit=2,
    gradient_accumulation_steps=4,
    optim='adamw_8bit',
    seed=42,
    report_to=[]
)

print("Training arguments configured")

## Step 10: Train Model

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

print("Starting instruction fine-tuning...")
train_result = trainer.train()
print(f"Training loss: {train_result.training_loss:.4f}")

## Step 11: Save Model

In [ ]:
# Save adapter
sft_adapter_path = './models/sft_adapter'
os.makedirs(sft_adapter_path, exist_ok=True)
model.save_pretrained(sft_adapter_path)
tokenizer.save_pretrained(sft_adapter_path)

print(f"SFT adapter saved to {sft_adapter_path}")

## Step 12: Test SFT Model

In [ ]:
def generate_response(instruction, max_length=150):
    prompt = f"### Instruction:
{instruction}
### Response:
"
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        inputs.input_ids,
        max_length=max_length,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test on sample questions
test_questions = [
    'What is machine learning?',
    'Explain gradient descent',
    'What is a neural network?'
]

print("Testing SFT fine-tuned model:\n")
for q in test_questions:
    print(f"Question: {q}")
    print(f"Answer: {generate_response(q)}")
    print()